# GPT-2 (124M) CPT on Indian legal data
Mirrors the Mamba-2 130M notebook: same dataset, same dev/test exclusion, same dedup, same tokens per step (32,768), causal LM objective.

In [10]:
import zipfile, math, os, time, json, hashlib, pickle, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, GPT2LMHeadModel, get_cosine_schedule_with_warmup

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130 | CUDA: True
GPU: NVIDIA RTX 4000 Ada Generation


In [11]:
MODEL_NAME = "gpt2"
DATASET_DIR = Path("Dataset")
OUTPUT_DIR = Path("./gpt2-124m-cpt-full")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OLD_EVAL_IDS = Path("./mamba2-130m-cpt-full/eval_ids.pkl")   # reused if present (tokenizer-independent)

NUM_ROWS = None       # None = all rows (use 500 for a pilot run)
MAX_STEPS = None     # None = full run (use 30 for a pilot run)
USE_FILENAME_KEY = True
TOKENS_PATH = OUTPUT_DIR / f"tokens_{NUM_ROWS or 'all'}.bin"

TRAIN_CSVS = [
    ("binary_multi_train.zip", "binary_multi_train/CJPE_ext_SCI_HCs_Tribunals_daily_orders_multi.csv"),
]

BLOCK_SIZE = 1024       # GPT-2's maximum context (n_positions)
MICRO_BATCH = 4         # lower if OOM
GRAD_ACCUM = 8          # 32 seqs/step * 1024 = 32,768 tokens/step (same as Mamba run)
EPOCHS = 1
LEARNING_RATE = 5e-5
WARMUP_FRAC = 0.03
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

LOG_EVERY = 10
EVAL_EVERY = 500
SAVE_EVERY = 500
SEED = 42
USE_BF16 = True

torch.manual_seed(SEED)

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.model_max_length = int(1e9)   # silence "sequence too long" warning (we chunk ourselves)

eos_id = tokenizer.eos_token_id          # <|endoftext|> = 50256
assert eos_id is not None and len(tokenizer) < 65536
print("Vocab:", len(tokenizer), "| EOS:", tokenizer.eos_token, eos_id)

Vocab: 50257 | EOS: <|endoftext|> 50256


In [13]:
def doc_key(t):
    return hashlib.md5(" ".join(t.lower().split()).encode("utf-8")).digest()

def iter_chunks(zip_name, csv_name, chunksize=2000):
    with zipfile.ZipFile(DATASET_DIR / zip_name) as z, z.open(csv_name) as f:
        for chunk in pd.read_csv(f, usecols=["filename", "text"], chunksize=chunksize):
            chunk = chunk.dropna(subset=["text"])
            chunk["text"] = chunk["text"].astype(str).str.strip()
            chunk["filename"] = chunk["filename"].astype(str)
            yield chunk[chunk["text"] != ""]

# ---- 1) dev/test documents to exclude from CPT (same set as the Mamba run) ----
EVAL_IDS_PATH = OUTPUT_DIR / "eval_ids.pkl"
if not EVAL_IDS_PATH.exists() and OLD_EVAL_IDS.exists():
    shutil.copy(OLD_EVAL_IDS, EVAL_IDS_PATH)

if EVAL_IDS_PATH.exists():
    with open(EVAL_IDS_PATH, "rb") as f:
        eval_files, eval_hashes = pickle.load(f)
else:
    eval_files, eval_hashes = set(), set()
    for zp in sorted(DATASET_DIR.glob("*.zip")):
        if not any(k in zp.name.lower() for k in ("dev", "test")):
            continue
        with zipfile.ZipFile(zp) as z:
            names = [n for n in z.namelist() if n.lower().endswith(".csv")]
        for n in names:
            for chunk in iter_chunks(zp.name, n):
                eval_files.update(chunk["filename"])
                eval_hashes.update(doc_key(t) for t in chunk["text"])
            print(f"[eval] {zp.name}:{n} | files {len(eval_files):,} | docs {len(eval_hashes):,}")
    with open(EVAL_IDS_PATH, "wb") as f:
        pickle.dump((eval_files, eval_hashes), f)
print(f"Eval documents excluded from CPT: {len(eval_hashes):,}")

# ---- 2) stream train CSVs -> dedup -> BPE tokenize -> uint16 file (EOS after each doc) ----
if TOKENS_PATH.exists():
    print(f"{TOKENS_PATH.name} exists - skipping tokenization (delete it to rebuild).")
else:
    seen_files, seen_hashes = set(), set()
    n_docs = n_tokens = skipped_eval = skipped_dup = 0
    done = False
    tmp_path = TOKENS_PATH.with_suffix(".tmp")
    with open(tmp_path, "wb") as out:
        for zip_name, csv_name in TRAIN_CSVS:
            for chunk in iter_chunks(zip_name, csv_name):
                kept = []
                for t, nme in zip(chunk["text"].tolist(), chunk["filename"].tolist()):
                    k = doc_key(t)
                    if k in eval_hashes or (USE_FILENAME_KEY and nme in eval_files):
                        skipped_eval += 1
                        continue
                    if k in seen_hashes or (USE_FILENAME_KEY and nme in seen_files):
                        skipped_dup += 1
                        continue
                    seen_hashes.add(k)
                    seen_files.add(nme)
                    kept.append(t)
                if NUM_ROWS is not None:
                    kept = kept[: NUM_ROWS - n_docs]
                if kept:
                    enc = tokenizer(kept, add_special_tokens=False, truncation=False)["input_ids"]
                    arr = np.concatenate([np.asarray(ids + [eos_id], dtype=np.uint16) for ids in enc])
                    arr.tofile(out)
                    n_docs += len(kept)
                    n_tokens += len(arr)
                print(f"  docs {n_docs:,} | tokens {n_tokens:,} | skipped eval {skipped_eval:,} dup {skipped_dup:,}", end="\r")
                if NUM_ROWS is not None and n_docs >= NUM_ROWS:
                    done = True
                    break
            if done:
                break
    os.replace(tmp_path, TOKENS_PATH)
    stats = dict(unique_docs=n_docs, total_tokens=n_tokens,
                 skipped_in_dev_test=skipped_eval, skipped_duplicates=skipped_dup)
    TOKENS_PATH.with_suffix(".json").write_text(json.dumps(stats, indent=2))
    print("\n", stats)

# ---- 3) memmap blocks + train/val split ----
data = np.memmap(TOKENS_PATH, dtype=np.uint16, mode="r")
n_blocks = len(data) // BLOCK_SIZE
blocks = data[: n_blocks * BLOCK_SIZE].reshape(n_blocks, BLOCK_SIZE)

def get_blocks(ids):
    ids = np.sort(np.asarray(ids))
    return torch.from_numpy(blocks[ids].astype(np.int64))

SEG = 16
n_seg = max(1, min(512, n_blocks // 20) // SEG)
starts = np.linspace(0, n_blocks - SEG, n_seg).astype(int)
val_np = np.unique(np.concatenate([np.arange(s, s + SEG) for s in starts]))
mask = np.ones(n_blocks, dtype=bool)
mask[val_np] = False
val_ids = torch.from_numpy(val_np)
train_ids = torch.from_numpy(np.flatnonzero(mask))
print(f"Blocks: {n_blocks:,} | train {len(train_ids):,} | val {len(val_ids):,} "
      f"| train tokens {len(train_ids) * BLOCK_SIZE:,}")

Eval documents excluded from CPT: 500,245
  docs 114,248 | tokens 294,779,930 | skipped eval 582,596 dup 6,101
 {'unique_docs': 114248, 'total_tokens': 294779930, 'skipped_in_dev_test': 582596, 'skipped_duplicates': 6101}
Blocks: 287,871 | train 287,359 | val 512 | train tokens 294,255,616


In [14]:
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, dtype=torch.float32)
model.to(device)
model.train()
model.config.use_cache = False
print("Parameters:", sum(p.numel() for p in model.parameters()))

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 431.12it/s]


Parameters: 124439808


In [15]:
decay, no_decay = [], []
for p in model.parameters():
    if p.requires_grad:
        (decay if p.ndim >= 2 else no_decay).append(p)

optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": WEIGHT_DECAY},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=LEARNING_RATE, betas=(0.9, 0.95),
)

seqs_per_step = MICRO_BATCH * GRAD_ACCUM
steps_per_epoch = len(train_ids) // seqs_per_step
total_steps = steps_per_epoch * EPOCHS
if MAX_STEPS is not None:
    total_steps = min(total_steps, MAX_STEPS)

scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=max(1, int(WARMUP_FRAC * total_steps)),
    num_training_steps=total_steps,
)
print(f"Steps/epoch: {steps_per_epoch:,} | total steps: {total_steps:,} | "
      f"tokens/step: {seqs_per_step * BLOCK_SIZE:,}")

Steps/epoch: 8,979 | total steps: 8,979 | tokens/step: 32,768


In [16]:
autocast_ctx = lambda: torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=USE_BF16)

@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for i in range(0, len(val_ids), MICRO_BATCH * 2):
        x = get_blocks(val_ids[i:i + MICRO_BATCH * 2]).to(device)
        with autocast_ctx():
            out = model(input_ids=x, labels=x, use_cache=False)
        losses.append(out.loss.item())
    model.train()
    loss = float(np.mean(losses))
    return loss, math.exp(loss)

CKPT_PATH = OUTPUT_DIR / "ckpt.pt"

def save_checkpoint(step):
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "step": step},
               CKPT_PATH.with_suffix(".tmp"))
    os.replace(CKPT_PATH.with_suffix(".tmp"), CKPT_PATH)

start_step = 0
if CKPT_PATH.exists():
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    start_step = ckpt["step"]
    print(f"Resumed from step {start_step}")
    del ckpt

if start_step == 0:
    v_loss, v_ppl = evaluate()
    print(f"Baseline (before CPT) | val loss {v_loss:.4f} | ppl {v_ppl:.2f}")

Baseline (before CPT) | val loss 3.5260 | ppl 33.99


In [17]:
print("\nStarting CPT...\n")
log_path = OUTPUT_DIR / "train_log.csv"
if start_step == 0:
    log_path.write_text("step,train_loss,lr,tokens_per_sec,val_loss,val_ppl\n")

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
run_loss, run_steps, run_tokens = 0.0, 0, 0
t_log = time.time()
cached_epoch, order = -1, None

for step in range(start_step, total_steps):
    epoch, j = divmod(step, steps_per_epoch)
    if epoch != cached_epoch:
        order = torch.randperm(len(train_ids), generator=torch.Generator().manual_seed(SEED + epoch))
        cached_epoch = epoch
    idx = order[j * seqs_per_step:(j + 1) * seqs_per_step]

    optimizer.zero_grad(set_to_none=True)
    step_loss = 0.0
    for m in range(GRAD_ACCUM):
        x = get_blocks(train_ids[idx[m * MICRO_BATCH:(m + 1) * MICRO_BATCH]]).to(device)
        with autocast_ctx():
            out = model(input_ids=x, labels=x, use_cache=False)
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        step_loss += loss.item()

    if not math.isfinite(step_loss):
        raise RuntimeError(f"Non-finite loss at step {step}. Try USE_BF16=False or a lower LEARNING_RATE.")

    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    optimizer.step()
    scheduler.step()

    run_loss += step_loss
    run_steps += 1
    run_tokens += seqs_per_step * BLOCK_SIZE
    done = step + 1

    if done % LOG_EVERY == 0 or done == total_steps:
        now = time.time()
        tps = run_tokens / (now - t_log)
        eta_min = (total_steps - done) * seqs_per_step * BLOCK_SIZE / tps / 60
        lr = scheduler.get_last_lr()[0]
        avg = run_loss / run_steps
        print(f"step {done}/{total_steps} | loss {avg:.4f} | lr {lr:.2e} | "
              f"{tps:,.0f} tok/s | ETA {eta_min:,.1f} min | "
              f"peak mem {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
        with open(log_path, "a") as f:
            f.write(f"{done},{avg:.5f},{lr:.3e},{tps:.0f},,\n")
        run_loss, run_steps, run_tokens, t_log = 0.0, 0, 0, now

    if done % EVAL_EVERY == 0 or done == total_steps:
        v_loss, v_ppl = evaluate()
        print(f"  >> val loss {v_loss:.4f} | ppl {v_ppl:.2f}")
        with open(log_path, "a") as f:
            f.write(f"{done},,,,{v_loss:.5f},{v_ppl:.3f}\n")

    if done % SAVE_EVERY == 0:
        save_checkpoint(done)

print(f"\nTraining finished in {(time.time() - t0) / 60:.1f} min")


Starting CPT...

step 10/8979 | loss 3.6714 | lr 1.86e-06 | 17,880 tok/s | ETA 273.9 min | peak mem 7.86 GB
step 20/8979 | loss 3.6430 | lr 3.72e-06 | 17,983 tok/s | ETA 272.1 min | peak mem 7.86 GB
step 30/8979 | loss 3.6293 | lr 5.58e-06 | 17,927 tok/s | ETA 272.6 min | peak mem 7.86 GB
step 40/8979 | loss 3.5642 | lr 7.43e-06 | 17,880 tok/s | ETA 273.0 min | peak mem 7.86 GB
step 50/8979 | loss 3.5284 | lr 9.29e-06 | 17,881 tok/s | ETA 272.7 min | peak mem 7.86 GB
step 60/8979 | loss 3.5059 | lr 1.12e-05 | 18,229 tok/s | ETA 267.2 min | peak mem 7.86 GB
step 70/8979 | loss 3.5062 | lr 1.30e-05 | 18,770 tok/s | ETA 259.2 min | peak mem 7.86 GB
step 80/8979 | loss 3.4604 | lr 1.49e-05 | 18,722 tok/s | ETA 259.6 min | peak mem 7.86 GB
step 90/8979 | loss 3.4225 | lr 1.67e-05 | 18,691 tok/s | ETA 259.7 min | peak mem 7.86 GB
step 100/8979 | loss 3.4336 | lr 1.86e-05 | 18,673 tok/s | ETA 259.7 min | peak mem 7.86 GB
step 110/8979 | loss 3.4037 | lr 2.04e-05 | 18,678 tok/s | ETA 259.3 mi

In [18]:
final_dir = OUTPUT_DIR / "final"
final_dir.mkdir(exist_ok=True)
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print("Saved to", final_dir.resolve())
print("Peak GPU memory allocated:", round(torch.cuda.max_memory_allocated() / 1024**3, 2), "GB")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.23it/s]

Saved to C:\Users\HP\Desktop\2022IMT-061 Priya Keshri MTP\gpt2-124m-cpt-full\final
Peak GPU memory allocated: 7.86 GB
